In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import psutil

print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")

RAM: 31.3 GB


In [5]:
import os

print("Kaggle input folders:")
for item in os.listdir("/kaggle/input"):
    print(" -", item)

Kaggle input folders:
 - datasets


In [6]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".tsv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/dharmisapariya/dataset/test_source2.tsv
/kaggle/input/datasets/dharmisapariya/dataset/test_source3.tsv
/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv
/kaggle/input/datasets/dharmisapariya/dataset/train_source3.tsv
/kaggle/input/datasets/dharmisapariya/dataset/test_source1.tsv
/kaggle/input/datasets/dharmisapariya/dataset/train_source2.tsv
/kaggle/input/datasets/dharmisapariya/dataset/train_source1.tsv


In [13]:
import pandas as pd
import numpy as np
import sklearn
import rapidfuzz
import jellyfish

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("rapidfuzz:", rapidfuzz.__version__)
print("jellyfish: installed")

pandas: 2.3.3
numpy: 2.0.2
scikit-learn: 1.6.1
rapidfuzz: 3.14.6
jellyfish: installed


In [14]:
from pathlib import Path

code_dir = Path("/kaggle/working/code")
code_dir.mkdir(parents=True, exist_ok=True)

script = r'''
import sys
import re
import unicodedata
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/normalized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000


def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))

    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def normalize_series(series):
    return series.fillna("").astype(str).map(normalize_text)


def find_file(filename):
    matches = list(INPUT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(f"Could not find {filename}")

    return matches[0]


def process_file(filename):
    input_path = find_file(filename)

    output_name = filename.replace(".tsv", "_normalized.tsv")
    output_path = OUTPUT_DIR / output_name

    print(f"\nProcessing: {input_path}")
    print(f"Output:    {output_path}")

    first_chunk = True
    total = 0

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False
    ):

        chunk["business_name_normalized"] = normalize_series(
            chunk["business_name"]
        )

        chunk["business_address_normalized"] = normalize_series(
            chunk["business_address"]
        )

        chunk.to_csv(
            output_path,
            sep="\t",
            index=False,
            mode="w" if first_chunk else "a",
            header=first_chunk
        )

        first_chunk = False
        total += len(chunk)

        print(f"Processed {total:,} rows")

    print(f"Finished: {total:,} rows")


if __name__ == "__main__":

    if len(sys.argv) != 2:
        print("Usage:")
        print("python normalize_and_save.py train")
        print("python normalize_and_save.py test")
        sys.exit(1)

    split = sys.argv[1]

    if split == "train":
        files = [
            "train_source1.tsv",
            "train_source2.tsv",
            "train_source3.tsv"
        ]

    elif split == "test":
        files = [
            "test_source1.tsv",
            "test_source2.tsv",
            "test_source3.tsv"
        ]

    else:
        raise ValueError("Use train or test")

    for filename in files:
        process_file(filename)

    print("\nAll normalization complete.")
'''

script_path = code_dir / "normalize_and_save.py"
script_path.write_text(script)

print("Created:", script_path)

Created: /kaggle/working/code/normalize_and_save.py


In [15]:
!python /kaggle/working/code/normalize_and_save.py train


Processing: /kaggle/input/datasets/dharmisapariya/dataset/train_source1.tsv
Output:    /kaggle/working/normalized/train_source1_normalized.tsv
Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Processed 600,000 rows
Processed 700,000 rows
Processed 800,000 rows
Processed 900,000 rows
Processed 1,000,000 rows
Processed 1,100,000 rows
Processed 1,200,000 rows
Processed 1,300,000 rows
Processed 1,400,000 rows
Processed 1,500,000 rows
Processed 1,600,000 rows
Processed 1,700,000 rows
Processed 1,800,000 rows
Processed 1,900,000 rows
Processed 2,000,000 rows
Processed 2,100,000 rows
Processed 2,200,000 rows
Processed 2,206,821 rows
Finished: 2,206,821 rows

Processing: /kaggle/input/datasets/dharmisapariya/dataset/train_source2.tsv
Output:    /kaggle/working/normalized/train_source2_normalized.tsv
Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Processed 600,0

In [16]:
from pathlib import Path

normalized_dir = Path("/kaggle/working/normalized")

for f in sorted(normalized_dir.glob("*.tsv")):
    size_gb = f.stat().st_size / (1024**3)
    print(f"{f.name:40} {size_gb:.2f} GB")

train_source1_normalized.tsv             0.35 GB
train_source2_normalized.tsv             0.77 GB
train_source3_normalized.tsv             0.80 GB


In [18]:
import pandas as pd
from pathlib import Path

normalized_dir = Path("/kaggle/working/normalized")

for f in sorted(normalized_dir.glob("*.tsv")):
    df = pd.read_csv(f, sep="\t", nrows=3, dtype=str)
    print("\n" + "=" * 70)
    print(f.name)
    print(df.columns.tolist())
    print(df[[
        "entity_id",
        "business_name_normalized",
        "business_address_normalized",
        "country"
    ]])


train_source1_normalized.tsv
['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized']
      entity_id business_name_normalized  \
0  S1-925783039      orelee s barbershop   
1  S1-773889195              prime money   
2  S1-377745466             b retail inc   

            business_address_normalized country  
0  1795 westchester drive high point nc      US  
1         17560 ellis road tahlequah ok      US  
2     1712 montebello avenue phoenix az      US  

train_source2_normalized.tsv
['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized']
      entity_id   business_name_normalized  \
0  S2-166376419                        NaN   
1  S2-764573417  holloway peak inc seafood   
2  S2-639257739                        NaN   

                     business_address_normalized country  
0        kh no 570 13 new delhi west delhi delhi   India  
1                 

In [24]:
from pathlib import Path

code_dir = Path("/kaggle/working/code")
code_dir.mkdir(parents=True, exist_ok=True)

v9_code = r'''
import sys
import re
import math
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

BASE = Path("/kaggle/working")
NORM = BASE / "normalized"
OUT = BASE / "candidates"
OUT.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

# Maximum number of rows allowed behind one blocking key.
MAX_KEY_FREQ = 1200

# Maximum candidates retained for each S1.
MAX_CANDIDATES = 500

# Minimum token length for useful blocking.
MIN_TOKEN_LEN = 3

STOPWORDS = {
    "the", "and", "for", "inc", "llc", "ltd", "limited",
    "company", "corp", "corporation", "co", "corporate",
    "group", "services", "service", "solutions", "solution",
    "international", "india", "usa", "us"
}


def clean(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if x == "nan":
        return ""
    return x


def tokens(text):
    text = clean(text)
    if not text:
        return []

    return [
        t for t in text.split()
        if len(t) >= MIN_TOKEN_LEN and t not in STOPWORDS
    ]


def name_keys(name):
    ts = tokens(name)

    if not ts:
        return []

    keys = set()

    # Exact distinctive tokens.
    for t in ts:
        keys.add(("NT", t))

    # Prefixes of longer tokens.
    for t in ts:
        if len(t) >= 5:
            keys.add(("NP5", t[:5]))

        if len(t) >= 6:
            keys.add(("NP6", t[:6]))

    # Whole normalized name.
    if len(name) >= 5:
        keys.add(("NE", name))

    # Sorted-token representation helps with word-order changes.
    if len(ts) >= 2:
        sorted_name = " ".join(sorted(ts))
        keys.add(("NS", sorted_name))

    return keys


def address_keys(address):
    address = clean(address)

    if not address:
        return []

    ts = tokens(address)

    if not ts:
        return []

    keys = set()

    # Address number.
    m = re.search(r"\b\d{1,6}\b", address)
    number = m.group(0) if m else ""

    # Distinctive address tokens.
    for t in ts:
        if len(t) >= 4:
            keys.add(("AT", t))

        if len(t) >= 6:
            keys.add(("AP6", t[:6]))

    # Number + distinctive token.
    if number:
        for t in ts:
            if len(t) >= 4:
                keys.add(("AN", number + "|" + t))

    return keys


def all_keys(name, address, country):
    country = clean(country)

    keys = set()

    for k in name_keys(name):
        keys.add(("C", country) + k)

    for k in address_keys(address):
        keys.add(("C", country) + k)

    return keys


def read_source(path):
    return pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False
    )


def build_index(source_path):
    print(f"\nBuilding index: {source_path.name}")

    index = defaultdict(list)
    frequency = Counter()

    total = 0

    for chunk in pd.read_csv(
        source_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE
    ):

        for row in chunk.itertuples(index=False):
            name = clean(row.business_name_normalized)
            address = clean(row.business_address_normalized)
            country = clean(row.country)
            entity_id = row.entity_id

            keys = all_keys(name, address, country)

            for key in keys:
                frequency[key] += 1

            total += 1

        print(f"Scanned {total:,} rows")

    print(f"Unique keys before filtering: {len(frequency):,}")

    usable_keys = {
        key for key, freq in frequency.items()
        if freq <= MAX_KEY_FREQ
    }

    print(f"Usable keys: {len(usable_keys):,}")

    # Second pass: construct compact index.
    total = 0

    for chunk in pd.read_csv(
        source_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE
    ):

        for row in chunk.itertuples(index=False):
            name = clean(row.business_name_normalized)
            address = clean(row.business_address_normalized)
            country = clean(row.country)
            entity_id = row.entity_id

            for key in all_keys(name, address, country):
                if key in usable_keys:
                    index[key].append(entity_id)

            total += 1

        print(f"Indexed {total:,} rows")

    print(f"Final index keys: {len(index):,}")

    return index


def generate_for_source1(
    s1_path,
    s2_index,
    s3_index,
    output_path,
    sample_limit=None
):

    print(f"\nGenerating candidates from {s1_path.name}")

    total_s1 = 0
    total_candidates = 0

    recall_rows = []

    with open(output_path, "w") as out:

        out.write("source1_entity_id\tcandidate_entity_id\n")

        for chunk in pd.read_csv(
            s1_path,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            chunksize=CHUNK_SIZE
        ):

            for row in chunk.itertuples(index=False):

                s1_id = row.entity_id

                name = clean(row.business_name_normalized)
                address = clean(row.business_address_normalized)
                country = clean(row.country)

                keys = all_keys(name, address, country)

                # candidate -> number of independent keys shared
                scores = Counter()

                for key in keys:

                    for candidate in s2_index.get(key, ()):
                        scores[candidate] += 1

                    for candidate in s3_index.get(key, ()):
                        scores[candidate] += 1

                if scores:

                    # Rank by:
                    # 1. number of shared blocking keys
                    # 2. deterministic entity ID
                    ranked = sorted(
                        scores.items(),
                        key=lambda x: (-x[1], x[0])
                    )

                    candidates = [
                        candidate
                        for candidate, score in ranked[:MAX_CANDIDATES]
                    ]

                    for candidate in candidates:
                        out.write(f"{s1_id}\t{candidate}\n")

                    total_candidates += len(candidates)

                total_s1 += 1

                if total_s1 % 10_000 == 0:
                    avg = total_candidates / total_s1

                    print(
                        f"S1 processed: {total_s1:,} | "
                        f"avg candidates: {avg:.1f}"
                    )

                if sample_limit and total_s1 >= sample_limit:
                    print(f"\nSample limit reached: {sample_limit:,}")
                    return total_s1, total_candidates

    return total_s1, total_candidates


def main():

    if len(sys.argv) < 2:
        print("Usage:")
        print("python build_candidates_v9.py 50000")
        print("python build_candidates_v9.py full")
        sys.exit(1)

    mode = sys.argv[1]

    s1 = NORM / "train_source1_normalized.tsv"
    s2 = NORM / "train_source2_normalized.tsv"
    s3 = NORM / "train_source3_normalized.tsv"

    print("Building S2 index...")
    s2_index = build_index(s2)

    print("\nBuilding S3 index...")
    s3_index = build_index(s3)

    sample_limit = None

    if mode != "full":
        sample_limit = int(mode)

    output = OUT / f"raw_candidates_v9_{mode}.tsv"

    processed, candidates = generate_for_source1(
        s1,
        s2_index,
        s3_index,
        output,
        sample_limit
    )

    print("\n==============================")
    print("V9 COMPLETE")
    print("==============================")
    print(f"S1 processed: {processed:,}")
    print(f"Candidates:    {candidates:,}")

    if processed:
        print(
            f"Average candidates/S1: "
            f"{candidates / processed:.2f}"
        )

    print(f"Output: {output}")


if __name__ == "__main__":
    main()
'''

path = code_dir / "build_candidates_v9.py"
path.write_text(v9_code)

print("Created:")
print(path)

Created:
/kaggle/working/code/build_candidates_v9.py


In [25]:
!python /kaggle/working/code/build_candidates_v9.py 50000

Building S2 index...

Building index: train_source2_normalized.tsv
Scanned 100,000 rows
Scanned 200,000 rows
Scanned 300,000 rows
Scanned 400,000 rows
Scanned 500,000 rows
Scanned 600,000 rows
Scanned 700,000 rows
Scanned 800,000 rows
Scanned 900,000 rows
Scanned 1,000,000 rows
Scanned 1,100,000 rows
Scanned 1,200,000 rows
Scanned 1,300,000 rows
Scanned 1,400,000 rows
Scanned 1,500,000 rows
Scanned 1,600,000 rows
Scanned 1,700,000 rows
Scanned 1,800,000 rows
Scanned 1,900,000 rows
Scanned 2,000,000 rows
Scanned 2,100,000 rows
Scanned 2,200,000 rows
Scanned 2,300,000 rows
Scanned 2,400,000 rows
Scanned 2,500,000 rows
Scanned 2,600,000 rows
Scanned 2,700,000 rows
Scanned 2,800,000 rows
Scanned 2,900,000 rows
Scanned 3,000,000 rows
Scanned 3,100,000 rows
Scanned 3,200,000 rows
Scanned 3,300,000 rows
Scanned 3,400,000 rows
Scanned 3,500,000 rows
Scanned 3,600,000 rows
Scanned 3,700,000 rows
Scanned 3,800,000 rows
Scanned 3,900,000 rows
Scanned 4,000,000 rows
Scanned 4,100,000 rows
Scanned 

In [27]:
from pathlib import Path

matches = list(Path("/kaggle").rglob("train_ground_truth.tsv"))

print("Found:")
for p in matches:
    print(p)

Found:
/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv


In [28]:
import pandas as pd
from pathlib import Path

CAND = Path("/kaggle/working/candidates/raw_candidates_v9_50000.tsv")
GT = Path("/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv")

# Load ground truth
gt = pd.read_csv(
    GT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

# Only evaluate S1 IDs that C9 actually processed
cand_ids = set(
    pd.read_csv(
        CAND,
        sep="\t",
        dtype=str,
        usecols=["source1_entity_id"]
    )["source1_entity_id"].unique()
)

gt = gt[gt["source1_entity_id"].isin(cand_ids)].copy()

# Build true-match map
true_map = {}
singleton_count = 0

for row in gt.itertuples(index=False):
    s1 = row.source1_entity_id
    matched = str(row.matched_entity_ids).strip()

    if not matched:
        singleton_count += 1
        true_map[s1] = set()
    else:
        true_map[s1] = {
            x.strip()
            for x in matched.split(",")
            if x.strip()
        }

print("S1s evaluated:", len(true_map))
print("Singletons:", singleton_count)

# Track matches found by C9
found_map = {
    s1: set()
    for s1, true_ids in true_map.items()
    if true_ids
}

# Read candidate file in chunks
for chunk in pd.read_csv(
    CAND,
    sep="\t",
    dtype=str,
    chunksize=500_000
):
    for row in chunk.itertuples(index=False):
        s1 = row.source1_entity_id
        candidate = row.candidate_entity_id

        if s1 in found_map and candidate in true_map[s1]:
            found_map[s1].add(candidate)

# Calculate recall
total_true = 0
found_true = 0
missed_true = 0

for s1, true_ids in true_map.items():
    if not true_ids:
        continue

    total_true += len(true_ids)
    found_true += len(found_map[s1])
    missed_true += len(true_ids - found_map[s1])

recall = found_true / total_true if total_true else 0

print("\n" + "=" * 50)
print("C9 CANDIDATE RECALL")
print("=" * 50)
print(f"True matches:       {total_true:,}")
print(f"Found in candidates:{found_true:,}")
print(f"Missed:             {missed_true:,}")
print(f"Recall:             {recall:.4%}")
print("=" * 50)

S1s evaluated: 49996
Singletons: 2783

C9 CANDIDATE RECALL
True matches:       173,213
Found in candidates:159,970
Missed:             13,243
Recall:             92.3545%


In [29]:
import pandas as pd
from pathlib import Path

CAND = Path("/kaggle/working/candidates/raw_candidates_v9_50000.tsv")
GT = Path("/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv")

# Load normalized source data
S1 = pd.read_csv(
    "/kaggle/working/normalized/train_source1_normalized.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

S2 = pd.read_csv(
    "/kaggle/working/normalized/train_source2_normalized.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

S3 = pd.read_csv(
    "/kaggle/working/normalized/train_source3_normalized.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

gt = pd.read_csv(
    GT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

# Only first 50k S1s
sample_s1 = set(
    S1["entity_id"].head(50000)
)

gt = gt[gt["source1_entity_id"].isin(sample_s1)].copy()

# Build candidate lookup
candidate_pairs = set()

for chunk in pd.read_csv(
    CAND,
    sep="\t",
    dtype=str,
    chunksize=500_000
):
    candidate_pairs.update(
        zip(
            chunk["source1_entity_id"],
            chunk["candidate_entity_id"]
        )
    )

print("Candidate pairs loaded:", len(candidate_pairs))

# Build S2/S3 lookup
other = pd.concat(
    [
        S2[["entity_id", "business_name_normalized",
            "business_address_normalized", "country"]],
        S3[["entity_id", "business_name_normalized",
            "business_address_normalized", "country"]]
    ],
    ignore_index=True
)

other = other.set_index("entity_id")

s1_lookup = S1.set_index("entity_id")

missed_examples = []

for row in gt.itertuples(index=False):

    s1_id = row.source1_entity_id
    matched = str(row.matched_entity_ids).strip()

    if not matched:
        continue

    true_ids = [
        x.strip()
        for x in matched.split(",")
        if x.strip()
    ]

    for true_id in true_ids:

        if (s1_id, true_id) in candidate_pairs:
            continue

        if s1_id not in s1_lookup.index:
            continue

        if true_id not in other.index:
            continue

        a = s1_lookup.loc[s1_id]
        b = other.loc[true_id]

        missed_examples.append({
            "s1_id": s1_id,
            "true_id": true_id,
            "country": a["country"],
            "s1_name": a["business_name_normalized"],
            "true_name": b["business_name_normalized"],
            "s1_address": a["business_address_normalized"],
            "true_address": b["business_address_normalized"]
        })

        if len(missed_examples) >= 100:
            break

    if len(missed_examples) >= 100:
        break

missed_df = pd.DataFrame(missed_examples)

print("\nMissed examples:", len(missed_df))
display(missed_df)

Candidate pairs loaded: 24104327

Missed examples: 100


,s1_id,true_id,country,s1_name,true_name,s1_address,true_address
0,S1-47773771,S3-514311755,US,primary care specialists inc,belocalo,141 08 71 road flushing ny,141 08b 71 road flushign new york
1,S1-47773771,S3-783274386,US,primary care specialists inc,primary care specialists incorporated,141 08 71 road flushing ny,141 08b 71 rd flushign new york
2,S1-47773771,S3-312828212,US,primary care specialists inc,primary care stpianlsts inc,141 08 71 road flushing ny,141 08b 71 rd flushign new york
3,S1-615771798,S2-46957492,US,straight edge hypnosis,straight edge hpyosris,decatur ar 16613 ar 102 highway,0016613 ar 102 hwy ar decatur
4,S1-991226123,S2-234928803,India,one infra private limited,,b 803 safal solitaire corporate park nr divya ...,b 8 03 ahmedabad hq region ahmedabad gujarat
...,...,...,...,...,...,...,...
95,S1-936282065,S2-808119943,India,creative impex private limited,creative impex pvrieae limited,f no 701 bld g 9 s no 612 613 615 ganga dham i...,h no 802 f no 701 bld g 9 s no 612 613 615 gan...
96,S1-936282065,S2-893779629,India,creative impex private limited,creative impex private,f no 701 bld g 9 s no 612 613 615 ganga dham i...,no 802 f no 701 bld g 9 s no 612 613 615 ganga...
97,S1-494307520,S2-75010030,US,supreme reserve group,supreme rseve group,107 pineneedle drive west end nc,
98,S1-429813907,S3-750318361,US,synex,ssyx,504 canaan road marshall ar,canaan road marshall arkansas


In [30]:
from rapidfuzz.fuzz import ratio, WRatio

# Compare the 100 missed examples using fuzzy similarity
results = []

for row in missed_df.itertuples(index=False):

    name_score = WRatio(
        str(row.s1_name),
        str(row.true_name)
    )

    address_score = WRatio(
        str(row.s1_address),
        str(row.true_address)
    )

    results.append({
        "s1_id": row.s1_id,
        "true_id": row.true_id,
        "name_score": round(name_score, 1),
        "address_score": round(address_score, 1),
        "s1_name": row.s1_name,
        "true_name": row.true_name,
        "s1_address": row.s1_address,
        "true_address": row.true_address
    })

fuzzy_df = pd.DataFrame(results)

print("Name score distribution:")
print(fuzzy_df["name_score"].describe())

print("\nAddress score distribution:")
print(fuzzy_df["address_score"].describe())

print("\nLowest name scores:")
display(
    fuzzy_df.sort_values("name_score").head(20)
)

print("\nHighest name scores:")
display(
    fuzzy_df.sort_values("name_score", ascending=False).head(20)
)

Name score distribution:
count    100.000000
mean      71.118000
std       32.807197
min        0.000000
25%       70.000000
50%       85.150000
75%       93.425000
max      100.000000
Name: name_score, dtype: float64

Address score distribution:
count    100.000000
mean      67.449000
std       34.785093
min        0.000000
25%       63.525000
50%       85.500000
75%       85.675000
max       98.200000
Name: address_score, dtype: float64

Lowest name scores:


,s1_id,true_id,name_score,address_score,s1_name,true_name,s1_address,true_address
4,S1-991226123,S2-234928803,0.0,85.5,one infra private limited,,b 803 safal solitaire corporate park nr divya ...,b 8 03 ahmedabad hq region ahmedabad gujarat
10,S1-951159035,S2-417819840,0.0,85.5,sai construction,,5th floor bbr avenue plot no 38 c block 7 3 br...,no g 5th floor rajendranagar k v rangareddy te...
12,S1-951159035,S3-723985585,0.0,85.5,sai construction,,5th floor bbr avenue plot no 38 c block 7 3 br...,5th floor rajendranagar k v rangareddy tg
17,S1-426267364,S2-380593256,0.0,85.5,prime projects private limited,,s o m vasanthan no 10 west karikalan street ad...,s o m vasanthan kancheepuram
27,S1-749410612,S3-264717775,0.0,96.1,raj investment private limited,,camp com the boulevard opp western bus hub sur...,camp com the boulevard opp western bus hub sur...
42,S1-526056745,S2-706207344,0.0,89.6,al ram engineering private limited,,400 3 giri market ghaziabad uttar pradesh,00 3 ghaziabad uttar pradesh
43,S1-526056745,S3-924222833,0.0,85.5,al ram engineering private limited,,400 3 giri market ghaziabad uttar pradesh,400 3 ghaziabad up
36,S1-88833811,S2-819207964,0.0,60.0,jay city builders,,delhi ohini 311 delhi 3rd floor vikas surya sh...,311 delhi
41,S1-526056745,S2-104088774,0.0,85.5,al ram engineering private limited,,400 3 giri market ghaziabad uttar pradesh,00 3 ghaziabad
59,S1-845929181,S3-100270922,0.0,85.5,vijay trading private limited,,j 3 299 ground floor dda flats kalka ji new de...,j 3 299 new delhi dl



Highest name scores:


,s1_id,true_id,name_score,address_score,s1_name,true_name,s1_address,true_address
26,S1-929436188,S3-601124866,100.0,0.0,bangalore south design private limited,bangalore south design private limited,no 189 4th cross 4th main dollars colony banga...,
48,S1-846680700,S3-544925633,100.0,85.5,chennai engineering private limited,chennai engineering private limited,4 12 ground floor rams chavali enclave door no...,4 12 chennai tn
34,S1-918839632,S3-11291190,98.7,85.5,services gain natural private limited,services gamin natural private limited,c 605 pearl rajhans dreams stella umele bassei...,no 808 c 605 thane
74,S1-196309581,S3-361204897,98.3,85.5,ua investments private limited,ua investments privae limited,616 swati crimson and clover nr shilaj circle ...,h no 176 616 ahmedabad daskroi gj
63,S1-886890337,S2-845448276,97.9,0.0,deu network corporation,deu netwaork corporation,22 surya sen pally rabindranagar kolkata howra...,
20,S1-890009542,S3-203500929,97.3,83.1,sj ace vendome inc,sj ace venhdome inc,60 uneeda street madison wv,west virginia 1 uneeda street madison
49,S1-846680700,S3-860720225,97.1,85.5,chennai engineering private limited,chennai engineering prviate limited,4 12 ground floor rams chavali enclave door no...,4 12 chennai
64,S1-747755104,S3-443774679,95.5,90.7,bhubaneswar mills llp,bhubaneswar mills l l p,b 57 saheed nagar bhubaneswar khordha orissa,b 1 57 saheed nagar bhubaneswar khordha od
78,S1-330702069,S2-676924412,95.2,85.5,bombay energy pvt ltd,8ombay energy pvt ltd,303 shankar marg hanuman nagar extension sirsi...,303 jaipur
31,S1-442165268,S3-404110097,95.0,71.6,legacy american pllc,legacy american,2480 old belfair highway belfair wa,washington old belfair hwy belfair


In [31]:
print("=== FUZZY RECOVERY POTENTIAL ===")

for threshold in [70, 75, 80, 85, 90, 95]:
    name_recover = (fuzzy_df["name_score"] >= threshold).sum()
    addr_recover = (fuzzy_df["address_score"] >= threshold).sum()

    either = (
        (fuzzy_df["name_score"] >= threshold) |
        (fuzzy_df["address_score"] >= threshold)
    ).sum()

    both = (
        (fuzzy_df["name_score"] >= threshold) &
        (fuzzy_df["address_score"] >= threshold)
    ).sum()

    print(
        f"Threshold {threshold}: "
        f"name={name_recover}/100, "
        f"address={addr_recover}/100, "
        f"either={either}/100, "
        f"both={both}/100"
    )

=== FUZZY RECOVERY POTENTIAL ===
Threshold 70: name=76/100, address=73/100, either=97/100, both=52/100
Threshold 75: name=67/100, address=71/100, either=93/100, both=45/100
Threshold 80: name=63/100, address=67/100, either=91/100, both=39/100
Threshold 85: name=50/100, address=58/100, either=84/100, both=24/100
Threshold 90: name=41/100, address=14/100, either=48/100, both=7/100
Threshold 95: name=22/100, address=10/100, either=29/100, both=3/100


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import vstack

# ============================================================
# C10 CONFIG
# ============================================================

SAMPLE_SIZE = 50_000
TOP_NAME = 10
TOP_ADDRESS = 10

S1_PATH = "/kaggle/working/normalized/train_source1_normalized.tsv"
S2_PATH = "/kaggle/working/normalized/train_source2_normalized.tsv"
S3_PATH = "/kaggle/working/normalized/train_source3_normalized.tsv"

OUT = Path("/kaggle/working/candidates")
OUT.mkdir(exist_ok=True)

# ============================================================
# LOAD FIRST 50K S1
# ============================================================

s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    nrows=SAMPLE_SIZE,
    keep_default_na=False
)

s1["business_name_normalized"] = (
    s1["business_name_normalized"].fillna("").astype(str)
)

s1["business_address_normalized"] = (
    s1["business_address_normalized"].fillna("").astype(str)
)

print("S1 loaded:", len(s1))

# ============================================================
# LOAD S2 + S3
# ============================================================

cols = [
    "entity_id",
    "business_name_normalized",
    "business_address_normalized",
    "country"
]

s2 = pd.read_csv(
    S2_PATH,
    sep="\t",
    dtype=str,
    usecols=cols,
    keep_default_na=False
)

s3 = pd.read_csv(
    S3_PATH,
    sep="\t",
    dtype=str,
    usecols=cols,
    keep_default_na=False
)

other = pd.concat([s2, s3], ignore_index=True)

del s2, s3

other["business_name_normalized"] = (
    other["business_name_normalized"].fillna("").astype(str)
)

other["business_address_normalized"] = (
    other["business_address_normalized"].fillna("").astype(str)
)

print("S2 + S3 records:", len(other))

# ============================================================
# COUNTRY RESTRICTION
# ============================================================
# Keep candidate retrieval within the same country.
# This prevents obviously unrelated cross-country retrieval.

# We process country groups separately so the TF-IDF search
# does not compare US records against India records.

all_results = []

countries = s1["country"].unique()

print("\nCountries:", countries)

for country in countries:

    print("\nProcessing country:", country)

    s1_mask = s1["country"] == country
    other_mask = other["country"] == country

    q = s1.loc[
        s1_mask,
        [
            "entity_id",
            "business_name_normalized",
            "business_address_normalized"
        ]
    ].copy()

    db = other.loc[
        other_mask,
        [
            "entity_id",
            "business_name_normalized",
            "business_address_normalized"
        ]
    ].copy()

    print("Queries:", len(q))
    print("Database:", len(db))

    if len(q) == 0 or len(db) == 0:
        continue

    # ========================================================
    # NAME TF-IDF
    # ========================================================

    print("Building name TF-IDF...")

    name_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
        dtype=np.float32
    )

    db_name = name_vectorizer.fit_transform(
        db["business_name_normalized"]
    )

    q_name = name_vectorizer.transform(
        q["business_name_normalized"]
    )

    print("Name matrix:", db_name.shape)

    name_nn = NearestNeighbors(
        n_neighbors=min(TOP_NAME, len(db)),
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    name_nn.fit(db_name)

    name_dist, name_idx = name_nn.kneighbors(q_name)

    del name_nn, db_name, q_name, name_vectorizer

    # ========================================================
    # ADDRESS TF-IDF
    # ========================================================

    print("Building address TF-IDF...")

    addr_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
        dtype=np.float32
    )

    db_addr = addr_vectorizer.fit_transform(
        db["business_address_normalized"]
    )

    q_addr = addr_vectorizer.transform(
        q["business_address_normalized"]
    )

    print("Address matrix:", db_addr.shape)

    addr_nn = NearestNeighbors(
        n_neighbors=min(TOP_ADDRESS, len(db)),
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    addr_nn.fit(db_addr)

    addr_dist, addr_idx = addr_nn.kneighbors(q_addr)

    del addr_nn, db_addr, q_addr, addr_vectorizer

    # ========================================================
    # COMBINE RESULTS
    # ========================================================

    for i, s1_id in enumerate(q["entity_id"]):

        candidate_ids = set()

        # Name candidates
        for j in name_idx[i]:
            candidate_ids.add(db.iloc[j]["entity_id"])

        # Address candidates
        for j in addr_idx[i]:
            candidate_ids.add(db.iloc[j]["entity_id"])

        for candidate_id in candidate_ids:

            all_results.append({
                "source1_entity_id": s1_id,
                "candidate_entity_id": candidate_id
            })

    del q, db, name_idx, name_dist, addr_idx, addr_dist

    print("Finished:", country)

# ============================================================
# SAVE TF-IDF SAFETY-NET CANDIDATES
# ============================================================

tfidf_candidates = pd.DataFrame(all_results)

tfidf_candidates = tfidf_candidates.drop_duplicates()

tfidf_candidates.to_csv(
    OUT / "tfidf_candidates_c10_50000.tsv",
    sep="\t",
    index=False
)

print("\n" + "=" * 60)
print("C10 TF-IDF CANDIDATES")
print("=" * 60)
print("Rows:", len(tfidf_candidates))
print("Unique S1:", tfidf_candidates["source1_entity_id"].nunique())
print(
    "Average candidates/S1:",
    len(tfidf_candidates) / SAMPLE_SIZE
)
print("=" * 60)

S1 loaded: 50000
S2 + S3 records: 10320219

Countries: ['US' 'India']

Processing country: US
Queries: 29965
Database: 6186873
Building name TF-IDF...
Name matrix: (6186873, 1387571)


In [1]:
from pathlib import Path

print("C9 index files:")
for p in Path("/kaggle/working").rglob("*"):
    if p.is_file() and ("index" in p.name.lower() or "candidate" in p.name.lower()):
        print(p)

C9 index files:
/kaggle/working/code/build_candidates_v9.py
/kaggle/working/candidates/raw_candidates_v9_50000.tsv


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

SAMPLE_SIZE = 50_000
TOP_NAME = 10
TOP_ADDRESS = 10
CHUNK_SIZE = 100_000

S1_PATH = "/kaggle/working/normalized/train_source1_normalized.tsv"
S2_PATH = "/kaggle/working/normalized/train_source2_normalized.tsv"
S3_PATH = "/kaggle/working/normalized/train_source3_normalized.tsv"

OUT_DIR = Path("/kaggle/working/candidates")
OUT_DIR.mkdir(exist_ok=True)

OUT = OUT_DIR / "tfidf_candidates_c10_50000.tsv"

# Load S1 sample
s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    nrows=SAMPLE_SIZE,
    keep_default_na=False
)

s1["business_name_normalized"] = (
    s1["business_name_normalized"]
    .fillna("")
    .replace("nan", "")
)

s1["business_address_normalized"] = (
    s1["business_address_normalized"]
    .fillna("")
    .replace("nan", "")
)

print("S1 loaded:", len(s1))

# Load S2/S3 IDs and country only first
cols = [
    "entity_id",
    "business_name_normalized",
    "business_address_normalized",
    "country"
]

print("Starting chunked C10...")

S1 loaded: 50000
Starting chunked C10...


In [5]:
from pathlib import Path

for p in [
    "/kaggle/working/normalized/train_source1_normalized.tsv",
    "/kaggle/working/normalized/train_source2_normalized.tsv",
    "/kaggle/working/normalized/train_source3_normalized.tsv",
    "/kaggle/working/candidates/raw_candidates_v9_50000.tsv"
]:
    path = Path(p)
    print("OK" if path.exists() else "MISSING", p)

OK /kaggle/working/normalized/train_source1_normalized.tsv
OK /kaggle/working/normalized/train_source2_normalized.tsv
OK /kaggle/working/normalized/train_source3_normalized.tsv
OK /kaggle/working/candidates/raw_candidates_v9_50000.tsv


In [2]:
from pathlib import Path

p = Path("/kaggle/working/candidates/raw_candidates_v9_50000.tsv")

print("C9:", "SAFE" if p.exists() else "MISSING")

if p.exists():
    print(f"Size: {p.stat().st_size / (1024**3):.2f} GB")

C9: SAFE
Size: 0.58 GB


In [9]:
import sys

# Remove partially loaded pandas modules
for name in list(sys.modules):
    if name == "pandas" or name.startswith("pandas."):
        del sys.modules[name]

print("Pandas modules cleared. Now importing fresh...")

import pandas as pd
print("Pandas OK:", pd.__version__)

Pandas modules cleared. Now importing fresh...
Pandas OK: 2.3.3


In [3]:
print("GT columns:", gt.columns.tolist())
print("C9 columns:", pd.read_csv(C9, sep="\t", dtype=str, nrows=3).columns.tolist())

print("\nGT sample:")
print(gt.head(3).to_string())

print("\nC9 sample:")
print(pd.read_csv(C9, sep="\t", dtype=str, nrows=5).to_string())

GT columns: ['source1_entity_id', 'matched_entity_ids']
C9 columns: ['source1_entity_id', 'candidate_entity_id']

GT sample:
  source1_entity_id                                               matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
1       S1-55344266              S2-249013014,S2-197070651,S3-478195123,S3-384364074
2      S1-343815751                           S2-790675320,S2-479876582,S3-878454467

C9 sample:
  source1_entity_id candidate_entity_id
0      S1-925783039        S2-157377754
1      S1-925783039        S2-517291332
2      S1-925783039        S3-997698194
3      S1-925783039        S2-335180728
4      S1-925783039        S3-698172821


In [4]:
# Get the actual S1 IDs contained in C9
c9_ids = set()

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    chunksize=500_000,
    keep_default_na=False
):
    c9_ids.update(chunk["source1_entity_id"].unique())

print("S1 IDs in C9:", len(c9_ids))

# Keep only those exact S1s from ground truth
gt_eval = gt[gt["source1_entity_id"].isin(c9_ids)].copy()

true_map = {}

for r in gt_eval.itertuples(index=False):
    ids = {
        x.strip()
        for x in r.matched_entity_ids.split(",")
        if x.strip()
    }
    true_map[r.source1_entity_id] = ids

found = {}

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    chunksize=500_000,
    keep_default_na=False
):
    for r in chunk.itertuples(index=False):
        if (
            r.source1_entity_id in true_map
            and r.candidate_entity_id in true_map[r.source1_entity_id]
        ):
            found.setdefault(r.source1_entity_id, set()).add(
                r.candidate_entity_id
            )

true_total = sum(len(x) for x in true_map.values())
found_total = sum(len(x) for x in found.values())

print("=" * 50)
print("CORRECTED C9 RECALL")
print("=" * 50)
print("Evaluated S1:", len(true_map))
print("True matches:", true_total)
print("Found:", found_total)
print("Missed:", true_total - found_total)
print("Recall:", round(found_total / true_total * 100, 4), "%")

S1 IDs in C9: 49996
CORRECTED C9 RECALL
Evaluated S1: 1132
True matches: 3848
Found: 3569
Missed: 279
Recall: 92.7495 %


In [5]:
import pandas as pd

GT = "/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv"

# Read complete ground truth
gt_all = pd.read_csv(
    GT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Total GT rows:", len(gt_all))
print("C9 S1 IDs:", len(c9_ids))

overlap = gt_all[gt_all["source1_entity_id"].isin(c9_ids)]

print("Exact overlap:", len(overlap))

Total GT rows: 2206821
C9 S1 IDs: 49996
Exact overlap: 49996


In [7]:
import pandas as pd

GT = "/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv"
C9 = "/kaggle/working/candidates/raw_candidates_v9_50000.tsv"

# Full ground truth
gt = pd.read_csv(
    GT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

# Build ground-truth map
true_map = {}

for r in gt.itertuples(index=False):
    true_map[r.source1_entity_id] = {
        x.strip()
        for x in r.matched_entity_ids.split(",")
        if x.strip()
    }

# Find candidates actually present in C9
found = {}

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    chunksize=500_000,
    keep_default_na=False
):
    for r in chunk.itertuples(index=False):
        if r.source1_entity_id in true_map:
            if r.candidate_entity_id in true_map[r.source1_entity_id]:
                found.setdefault(r.source1_entity_id, set()).add(
                    r.candidate_entity_id
                )

# Calculate missed TRUE links
missed_pairs = []

for s1_id, true_ids in true_map.items():
    for target_id in true_ids - found.get(s1_id, set()):
        missed_pairs.append((s1_id, target_id))

print("=" * 55)
print("C9 BLOCKING DIAGNOSTIC")
print("=" * 55)
print("Total S1:", len(true_map))
print("True matches:", sum(len(x) for x in true_map.values()))
print("Found:", sum(len(x) for x in found.values()))
print("Missed:", len(missed_pairs))
print(
    "Recall:",
    round(
        sum(len(x) for x in found.values())
        / sum(len(x) for x in true_map.values()) * 100,
        4
    ),
    "%"
)
print("Missed S1 entities:", len(set(x[0] for x in missed_pairs)))

C9 BLOCKING DIAGNOSTIC
Total S1: 2206821
True matches: 7638365
Found: 159970
Missed: 7478395
Recall: 2.0943 %
Missed S1 entities: 2045967


In [8]:
import pandas as pd
import os

GT = "/kaggle/input/datasets/dharmisapariya/dataset/train_ground_truth.tsv"
C9 = "/kaggle/working/candidates/raw_candidates_v9_50000.tsv"

OUT = "/kaggle/working/candidates/c9_missed_pairs.tsv"

print("=" * 60)
print("C9 CLEAN BLOCKING ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Fresh ground truth
# ------------------------------------------------------------

gt = pd.read_csv(
    GT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Ground truth rows:", len(gt))

# ------------------------------------------------------------
# 2. Get exact S1 population from C9
# ------------------------------------------------------------

c9_s1 = set()

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    chunksize=500_000,
    keep_default_na=False
):
    c9_s1.update(chunk["source1_entity_id"].unique())

print("C9 S1 IDs:", len(c9_s1))

# ------------------------------------------------------------
# 3. Restrict GT to exactly those S1s
# ------------------------------------------------------------

eval_gt = gt[
    gt["source1_entity_id"].isin(c9_s1)
].copy()

print("Evaluation S1:", len(eval_gt))

# ------------------------------------------------------------
# 4. Build true-match map
# ------------------------------------------------------------

true_map = {}

for r in eval_gt.itertuples(index=False):
    true_map[r.source1_entity_id] = {
        x.strip()
        for x in r.matched_entity_ids.split(",")
        if x.strip()
    }

true_total = sum(len(x) for x in true_map.values())

# ------------------------------------------------------------
# 5. Scan C9 and find true matches
# ------------------------------------------------------------

found = {}

rows_read = 0

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    chunksize=500_000,
    keep_default_na=False
):
    rows_read += len(chunk)

    for r in chunk.itertuples(index=False):

        true_ids = true_map.get(r.source1_entity_id)

        if true_ids is not None:
            if r.candidate_entity_id in true_ids:
                found.setdefault(
                    r.source1_entity_id,
                    set()
                ).add(r.candidate_entity_id)

# ------------------------------------------------------------
# 6. Calculate missed true links
# ------------------------------------------------------------

missed_rows = []

found_total = 0

for s1_id, true_ids in true_map.items():

    found_ids = found.get(s1_id, set())

    found_total += len(found_ids)

    for target_id in true_ids - found_ids:
        missed_rows.append({
            "source1_entity_id": s1_id,
            "missed_entity_id": target_id
        })

missed_total = len(missed_rows)

recall = (
    found_total / true_total * 100
    if true_total
    else 0
)

# ------------------------------------------------------------
# 7. Save missed pairs
# ------------------------------------------------------------

missed_df = pd.DataFrame(missed_rows)

missed_df.to_csv(
    OUT,
    sep="\t",
    index=False
)

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print()
print("=" * 60)
print("RESULT")
print("=" * 60)

print("C9 rows scanned       :", rows_read)
print("Evaluation S1         :", len(true_map))
print("True matches          :", true_total)
print("Found by C9           :", found_total)
print("Missed true matches   :", missed_total)
print("C9 blocking recall    :", round(recall, 4), "%")
print("Missed S1 entities    :", len(set(
    x["source1_entity_id"] for x in missed_rows
)))

print()
print("Saved missed pairs to:")
print(OUT)

print()
print("=" * 60)
print("FIRST 30 MISSED PAIRS")
print("=" * 60)

if missed_rows:
    print(missed_df.head(30).to_string(index=False))
else:
    print("No missed pairs.")

print()
print("DONE.")

C9 CLEAN BLOCKING ANALYSIS
Ground truth rows: 2206821
C9 S1 IDs: 49996
Evaluation S1: 49996

RESULT
C9 rows scanned       : 24104327
Evaluation S1         : 49996
True matches          : 173213
Found by C9           : 159970
Missed true matches   : 13243
C9 blocking recall    : 92.3545 %
Missed S1 entities    : 9606

Saved missed pairs to:
/kaggle/working/candidates/c9_missed_pairs.tsv

FIRST 30 MISSED PAIRS
source1_entity_id missed_entity_id
      S1-47773771     S3-514311755
      S1-47773771     S3-783274386
      S1-47773771     S3-312828212
     S1-615771798      S2-46957492
     S1-991226123     S2-234928803
     S1-991226123     S3-952060287
     S1-210849781     S3-240268161
     S1-705083256     S3-189320525
     S1-681160793     S2-891856879
      S1-59474645     S3-180857360
     S1-951159035     S3-723985585
     S1-951159035     S2-875950440
     S1-951159035     S2-417819840
     S1-958297307     S3-103154542
     S1-958297307     S2-753140238
     S1-958297307     S3-278

In [9]:
import pandas as pd
from rapidfuzz.fuzz import WRatio

S1 = "/kaggle/working/normalized/train_source1_normalized.tsv"
S2 = "/kaggle/working/normalized/train_source2_normalized.tsv"
S3 = "/kaggle/working/normalized/train_source3_normalized.tsv"

MISS = "/kaggle/working/candidates/c9_missed_pairs.tsv"

miss = pd.read_csv(
    MISS,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

s1_ids = set(miss["source1_entity_id"])
s2_ids = set(
    miss.loc[
        miss["missed_entity_id"].str.startswith("S2-"),
        "missed_entity_id"
    ]
)
s3_ids = set(
    miss.loc[
        miss["missed_entity_id"].str.startswith("S3-"),
        "missed_entity_id"
    ]
)

print("Missed pairs:", len(miss))
print("Unique S1:", len(s1_ids))
print("Unique S2 targets:", len(s2_ids))
print("Unique S3 targets:", len(s3_ids))


def clean(x):
    if pd.isna(x):
        return ""
    x = str(x)
    if x.lower() == "nan":
        return ""
    return x


# ------------------------------------------------------------
# Load only the relevant S1 records
# ------------------------------------------------------------

s1_parts = []

for chunk in pd.read_csv(
    S1,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    chunksize=100_000
):
    part = chunk[chunk["entity_id"].isin(s1_ids)]

    if len(part):
        s1_parts.append(part)

s1_df = pd.concat(s1_parts, ignore_index=True)

s1_df = s1_df[
    ["entity_id", "business_name_normalized",
     "business_address_normalized", "country"]
]

s1_df.columns = [
    "s1_id", "s1_name", "s1_address", "country"
]

# ------------------------------------------------------------
# Load only missed S2/S3 target records
# ------------------------------------------------------------

def load_targets(path, wanted):
    parts = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=100_000
    ):
        part = chunk[chunk["entity_id"].isin(wanted)]

        if len(part):
            parts.append(part)

    if not parts:
        return pd.DataFrame(
            columns=[
                "target_id",
                "target_name",
                "target_address"
            ]
        )

    out = pd.concat(parts, ignore_index=True)

    return out[
        ["entity_id",
         "business_name_normalized",
         "business_address_normalized"]
    ].rename(columns={
        "entity_id": "target_id",
        "business_name_normalized": "target_name",
        "business_address_normalized": "target_address"
    })


s2_df = load_targets(S2, s2_ids)
s3_df = load_targets(S3, s3_ids)

targets = pd.concat(
    [s2_df, s3_df],
    ignore_index=True
)

# ------------------------------------------------------------
# Join missed pairs
# ------------------------------------------------------------

pairs = miss.merge(
    s1_df,
    left_on="source1_entity_id",
    right_on="s1_id",
    how="left"
)

pairs = pairs.merge(
    targets,
    left_on="missed_entity_id",
    right_on="target_id",
    how="left"
)

# ------------------------------------------------------------
# Fuzzy diagnostics
# ------------------------------------------------------------

pairs["name_score"] = pairs.apply(
    lambda r: WRatio(
        clean(r["s1_name"]),
        clean(r["target_name"])
    ) if clean(r["s1_name"]) and clean(r["target_name"]) else 0,
    axis=1
)

pairs["address_score"] = pairs.apply(
    lambda r: WRatio(
        clean(r["s1_address"]),
        clean(r["target_address"])
    ) if clean(r["s1_address"]) and clean(r["target_address"]) else 0,
    axis=1
)

pairs["either_score"] = pairs[
    ["name_score", "address_score"]
].max(axis=1)

print()
print("=" * 60)
print("MISSED-PAIR DIAGNOSTIC")
print("=" * 60)

print(
    "Name >= 70:",
    (pairs["name_score"] >= 70).sum()
)

print(
    "Name >= 80:",
    (pairs["name_score"] >= 80).sum()
)

print(
    "Address >= 70:",
    (pairs["address_score"] >= 70).sum()
)

print(
    "Address >= 80:",
    (pairs["address_score"] >= 80).sum()
)

print(
    "Either >= 70:",
    (pairs["either_score"] >= 70).sum()
)

print(
    "Either >= 80:",
    (pairs["either_score"] >= 80).sum()
)

print()
print("Average name score:",
      round(pairs["name_score"].mean(), 2))

print("Average address score:",
      round(pairs["address_score"].mean(), 2))

print()
print("=" * 60)
print("20 MISSED PAIRS")
print("=" * 60)

cols = [
    "source1_entity_id",
    "missed_entity_id",
    "s1_name",
    "target_name",
    "name_score",
    "s1_address",
    "target_address",
    "address_score"
]

print(
    pairs[cols]
    .head(20)
    .to_string(index=False)
)

pairs.to_csv(
    "/kaggle/working/candidates/c9_missed_diagnostics.tsv",
    sep="\t",
    index=False
)

print()
print("Saved diagnostic file.")

Missed pairs: 13243
Unique S1: 9606
Unique S2 targets: 4979
Unique S3 targets: 8264

MISSED-PAIR DIAGNOSTIC
Name >= 70: 9799
Name >= 80: 8629
Address >= 70: 9809
Address >= 80: 8648
Either >= 70: 13064
Either >= 80: 12477

Average name score: 70.14
Average address score: 66.76

20 MISSED PAIRS
source1_entity_id missed_entity_id                        s1_name                           target_name  name_score                                                                                                     s1_address                                        target_address  address_score
      S1-47773771     S3-514311755   primary care specialists inc                              belocalo   45.000000                                                                                     141 08 71 road flushing ny                     141 08b 71 road flushign new york      84.745763
      S1-47773771     S3-783274386   primary care specialists inc primary care specialists incorporated   87.6923

In [10]:
import pandas as pd
from collections import defaultdict

S1 = "/kaggle/working/normalized/train_source1_normalized.tsv"
S2 = "/kaggle/working/normalized/train_source2_normalized.tsv"
S3 = "/kaggle/working/normalized/train_source3_normalized.tsv"

# Use the exact C9 validation S1 population
C9 = "/kaggle/working/candidates/raw_candidates_v9_50000.tsv"

c9_ids = set()

for chunk in pd.read_csv(
    C9,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    chunksize=500_000
):
    c9_ids.update(chunk["source1_entity_id"])

print("Validation S1:", len(c9_ids))


def clean(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if x == "nan":
        return ""
    return x


def block_key(value):
    """
    Character-level coarse block.

    First 2 characters + last 2 characters + length bucket.
    """
    value = clean(value)

    if len(value) < 4:
        return None

    length_bucket = len(value) // 3

    return (
        value[:2],
        value[-2:],
        length_bucket
    )


# ------------------------------------------------------------
# 1. Load only validation S1
# ------------------------------------------------------------

s1_parts = []

for chunk in pd.read_csv(
    S1,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    chunksize=100_000
):
    part = chunk[
        chunk["entity_id"].isin(c9_ids)
    ]

    if len(part):
        s1_parts.append(part)

s1 = pd.concat(
    s1_parts,
    ignore_index=True
)

s1 = s1[
    [
        "entity_id",
        "business_name_normalized",
        "business_address_normalized",
        "country"
    ]
]

print("Loaded S1:", len(s1))


# ------------------------------------------------------------
# 2. Create S1 block keys
# ------------------------------------------------------------

s1["name_block"] = s1[
    "business_name_normalized"
].map(block_key)

s1["address_block"] = s1[
    "business_address_normalized"
].map(block_key)


# ------------------------------------------------------------
# 3. Build lightweight S1 lookup
# ------------------------------------------------------------

name_queries = defaultdict(list)
address_queries = defaultdict(list)

for r in s1.itertuples(index=False):

    if r.name_block is not None:
        name_queries[
            (r.country, r.name_block)
        ].append(r.entity_id)

    if r.address_block is not None:
        address_queries[
            (r.country, r.address_block)
        ].append(r.entity_id)


print("Name blocks:", len(name_queries))
print("Address blocks:", len(address_queries))


# ------------------------------------------------------------
# 4. Scan S2/S3 and generate additional candidates
# ------------------------------------------------------------

additional = set()

def process_source(path):

    count = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=100_000
    ):

        for r in chunk.itertuples(index=False):

            country = clean(r.country)

            name = clean(r.business_name_normalized)
            address = clean(r.business_address_normalized)

            nk = block_key(name)
            ak = block_key(address)

            # Name block
            if nk is not None:
                for s1_id in name_queries.get(
                    (country, nk),
                    []
                ):
                    additional.add(
                        (
                            s1_id,
                            r.entity_id
                        )
                    )

            # Address block
            if ak is not None:
                for s1_id in address_queries.get(
                    (country, ak),
                    []
                ):
                    additional.add(
                        (
                            s1_id,
                            r.entity_id
                        )
                    )

        count += len(chunk)

        if count % 1_000_000 == 0:
            print(
                "Processed:",
                count,
                "additional candidates:",
                len(additional)
            )

    return count


print("\nScanning Source 2...")
process_source(S2)

print("\nScanning Source 3...")
process_source(S3)


print("\n" + "=" * 60)
print("ADDITIONAL BLOCKING RESULTS")
print("=" * 60)

print("Additional candidate pairs:", len(additional))

Validation S1: 49996
Loaded S1: 49996
Name blocks: 22755
Address blocks: 28394

Scanning Source 2...
Processed: 1000000 additional candidates: 0
Processed: 2000000 additional candidates: 0
Processed: 3000000 additional candidates: 0
Processed: 4000000 additional candidates: 0
Processed: 5000000 additional candidates: 0

Scanning Source 3...
Processed: 1000000 additional candidates: 0
Processed: 2000000 additional candidates: 0
Processed: 3000000 additional candidates: 0
Processed: 4000000 additional candidates: 0
Processed: 5000000 additional candidates: 0

ADDITIONAL BLOCKING RESULTS
Additional candidate pairs: 0
